In [1]:
# work environment: faiss_env
import pandas as pd
import numpy as np
import pickle
import sys
import os
import warnings
from pathlib import Path
from scipy.spatial import KDTree

gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="xarray")

from GEMS_TCO import configuration as config
from GEMS_TCO import data_preprocess as dmbh

# =====================================================================
# 정방향 Nearest Binning 함수
#
# [mean() → nearest 변경 이유]
# - 기존 mean(): k≥2 관측이 같은 셀에 들어오면 평균화 → effective nugget = σ²/k
#   → DW/Vecchia에서 nugget 과소추정
# - nearest: 셀 중심에 가장 가까운 관측 1개만 선택 → k=1 보장, nugget 온전히 보존
#
# [역방향 binning과의 비교]
# - 역방향(구버전): 격자 → 실측 (중복 배정 0.12% 발생 가능)
# - 정방향 nearest: 실측 → 격자, 셀당 nearest 1개 선택 → 중복 배정 0%
#   수학적으로 동일한 결과, 더 일관된 방향성
# =====================================================================
def forward_bin_for_whittle(loaded_map, base_center_points, step_lat, step_lon, v_drift_lon=-0.0048):
    """
    정방향 Nearest Binning으로 격자 데이터를 생성한다.
    각 격자 셀에 가장 가까운 관측 1개만 배정 (k=1 보장).

    Parameters
    ----------
    loaded_map        : dict, key=시간 key, value=DataFrame (실측 위성 데이터)
    base_center_points: array-like (n_grid, 2), 기준 격자 중심 [lat, lon]
    step_lat, step_lon: float, 격자 간격 (도 단위)
    v_drift_lon       : float, 시간대별 격자 서쪽 이동 속도 (도/시간)

    Returns
    -------
    coarse_cen_map : dict, 각 key에 대해 (n_grid, cols) DataFrame
                     배정된 실측 없는 격자 → 값 컬럼 NaN, 좌표는 base grid 고정
    """
    coarse_cen_map = {}
    sorted_keys = sorted(loaded_map.keys())

    lat_thresh = step_lat / 2.0
    lon_thresh = step_lon / 2.0

    if isinstance(base_center_points, pd.DataFrame):
        base_center_points = base_center_points.iloc[:, 0:2].values
    elif isinstance(base_center_points, list):
        base_center_points = np.array(base_center_points)

    n_grid = len(base_center_points)

    for i, key in enumerate(sorted_keys):
        t_idx = i % 8
        df_raw = loaded_map[key]

        if len(df_raw) == 0:
            continue

        shifted_grid = base_center_points + np.array([0.0, t_idx * v_drift_lon])

        if 'Latitude' in df_raw.columns and 'Longitude' in df_raw.columns:
            raw_coords = df_raw[['Latitude', 'Longitude']].values
        else:
            raw_coords = df_raw.iloc[:, 0:2].values

        value_cols = [c for c in df_raw.columns
                      if c not in ('Latitude', 'Longitude', 'FinalAlgorithmFlags')
                      and pd.api.types.is_numeric_dtype(df_raw[c])]

        grid_tree = KDTree(shifted_grid)
        _, grid_indices = grid_tree.query(raw_coords)

        lat_diffs = np.abs(raw_coords[:, 0] - shifted_grid[grid_indices, 0])
        lon_diffs = np.abs(raw_coords[:, 1] - shifted_grid[grid_indices, 1])
        valid = (lat_diffs <= lat_thresh) & (lon_diffs <= lon_thresh)

        df_valid = df_raw.loc[valid].copy()
        df_valid['_grid_idx'] = grid_indices[valid]
        df_valid['_src_lat']  = raw_coords[valid, 0]
        df_valid['_src_lon']  = raw_coords[valid, 1]

        # 셀 중심까지의 정규화 거리 계산 (lat/lon 스케일 통일)
        df_valid['_dist'] = np.hypot(
            (raw_coords[valid, 0] - shifted_grid[grid_indices[valid], 0]) / lat_thresh,
            (raw_coords[valid, 1] - shifted_grid[grid_indices[valid], 1]) / lon_thresh
        )

        # 각 격자 셀에서 가장 가까운 관측 1개만 선택 (k=1 보장)
        idx_nearest = df_valid.groupby('_grid_idx')['_dist'].idxmin()
        df_nearest = df_valid.loc[idx_nearest].set_index('_grid_idx')

        df_result = pd.DataFrame(
            np.nan, index=range(n_grid),
            columns=value_cols + ['Source_Latitude', 'Source_Longitude']
        )
        df_result.loc[df_nearest.index, value_cols]        = df_nearest[value_cols].values
        df_result.loc[df_nearest.index, 'Source_Latitude'] = df_nearest['_src_lat'].values
        df_result.loc[df_nearest.index, 'Source_Longitude'] = df_nearest['_src_lon'].values

        df_result.insert(0, 'Latitude',  base_center_points[:, 0])
        df_result.insert(1, 'Longitude', base_center_points[:, 1])

        coarse_cen_map[key] = df_result

    return coarse_cen_map
# =====================================================================


def process_coarse_data(base_path, years, months, lat_lon_bounds, step_sizes, base_csv_path, grid_type='rect'):
    print(f"\n--- Starting Forward Nearest Binning Process (Type: {grid_type}) ---")

    lat_start, lat_end, lon_start, lon_end = lat_lon_bounds
    step_lat, step_lon = step_sizes

    try:
        print(f"Loading base dataframe from: {base_csv_path}")
        df = pd.read_csv(base_csv_path)
        instance = dmbh.center_matching_hour(df, lat_start, lat_end, lon_start, lon_end)
    except Exception as e:
        print(f"Error initializing instance: {e}")
        return

    if grid_type == 'rect':
        print("Generating Rectangular Center Points (No Calibration)...")
        center_points = instance.make_center_points_wo_calibration(step_lat=step_lat, step_lon=step_lon)
    else:
        print("Generating Standard Center Points (With Calibration)...")
        center_points = instance.make_center_points(step_lat=step_lat, step_lon=step_lon)

    for year in years:
        for month in months:
            month_str = f"{month:02d}"
            print(f">> Processing: Year {year}, Month {month_str}")

            try:
                pickle_path     = os.path.join(base_path, f'pickle_{year}')
                input_filename  = f"orbit_map{str(year)[2:]}_{month_str}.pkl"
                output_filename = f"tco_grid_{str(year)[2:]}_{month_str}.pkl"

                input_filepath  = os.path.join(pickle_path, input_filename)
                output_filepath = os.path.join(pickle_path, output_filename)

                if not os.path.exists(input_filepath):
                    print(f"   [Skip] File not found: {input_filename}")
                    continue

                with open(input_filepath, 'rb') as f:
                    loaded_map = pickle.load(f)

                coarse_cen_map = forward_bin_for_whittle(
                    loaded_map=loaded_map,
                    base_center_points=center_points,
                    step_lat=step_lat,
                    step_lon=step_lon
                )

                os.makedirs(pickle_path, exist_ok=True)
                with open(output_filepath, 'wb') as f:
                    pickle.dump(coarse_cen_map, f)

                print(f"   [Saved] {output_filename}")

            except Exception as e:
                print(f"   [Error] {e}")
                import traceback; traceback.print_exc()


# ==========================================
# 실행부 (Main)
# ==========================================
if __name__ == '__main__':
    BASE_PATH = config.mac_data_load_path

    target_year  = 2022
    target_month = 7

    LAT_LON_BOUNDS = (-3, 2, 121, 131)
    STEP_SIZES     = (0.044, 0.063)

    BASE_CSV_PATH = (
        f"/Users/joonwonlee/Documents/GEMS_DATA/data_{target_year}/"
        f"data_{str(target_year)[2:]}_07_0131_N-32_E121131.csv"
    )

    process_coarse_data(
        base_path=BASE_PATH,
        years=[target_year],
        months=[target_month],
        lat_lon_bounds=LAT_LON_BOUNDS,
        step_sizes=STEP_SIZES,
        base_csv_path=BASE_CSV_PATH,
        grid_type='rect'
    )


--- Starting Forward Nearest Binning Process (Type: rect) ---
Loading base dataframe from: /Users/joonwonlee/Documents/GEMS_DATA/data_2022/data_22_07_0131_N-32_E121131.csv
Generating Rectangular Center Points (No Calibration)...
>> Processing: Year 2022, Month 07
   [Saved] tco_grid_22_07.pkl


In [2]:
# ==========================================
# 실행: 2022 / 2024 / 2025 July 재생성
# ==========================================
BASE_PATH      = config.mac_data_load_path
LAT_LON_BOUNDS = (-3, 2, 121, 131)
STEP_SIZES     = (0.044, 0.063)

for target_year in [2022, 2024, 2025]:
    yy = str(target_year)[2:]
    base_csv = (
        f"/Users/joonwonlee/Documents/GEMS_DATA/data_{target_year}/"
        f"data_{yy}_07_0131_N-32_E121131.csv"
    )
    print(f"\n{'='*50}")
    print(f"Year {target_year}")
    print(f"{'='*50}")
    process_coarse_data(
        base_path=BASE_PATH,
        years=[target_year],
        months=[7],
        lat_lon_bounds=LAT_LON_BOUNDS,
        step_sizes=STEP_SIZES,
        base_csv_path=base_csv,
        grid_type='rect'
    )


Year 2022

--- Starting Forward Nearest Binning Process (Type: rect) ---
Loading base dataframe from: /Users/joonwonlee/Documents/GEMS_DATA/data_2022/data_22_07_0131_N-32_E121131.csv
Generating Rectangular Center Points (No Calibration)...
>> Processing: Year 2022, Month 07
   [Saved] tco_grid_22_07.pkl

Year 2024

--- Starting Forward Nearest Binning Process (Type: rect) ---
Loading base dataframe from: /Users/joonwonlee/Documents/GEMS_DATA/data_2024/data_24_07_0131_N-32_E121131.csv
Generating Rectangular Center Points (No Calibration)...
>> Processing: Year 2024, Month 07
   [Saved] tco_grid_24_07.pkl

Year 2025

--- Starting Forward Nearest Binning Process (Type: rect) ---
Loading base dataframe from: /Users/joonwonlee/Documents/GEMS_DATA/data_2025/data_25_07_0131_N-32_E121131.csv
Generating Rectangular Center Points (No Calibration)...
>> Processing: Year 2025, Month 07
   [Saved] tco_grid_25_07.pkl


## 저장 결과 검증

In [ ]:
import sys, pickle, os
import numpy as np, pandas as pd
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)
from GEMS_TCO import configuration as config

for year, month in [(2022, 7), (2024, 7), (2025, 7)]:
    pickle_path = os.path.join(config.mac_data_load_path, f'pickle_{year}')
    fname = f"tco_grid_{str(year)[2:]}_{month:02d}.pkl"
    fpath = os.path.join(pickle_path, fname)

    if not os.path.exists(fpath):
        print(f"[{year}-{month:02d}] NOT FOUND: {fname}")
        continue

    with open(fpath, 'rb') as f:
        result = pickle.load(f)

    sample_key = sorted(result.keys())[0]
    df = result[sample_key]
    nan_rate = df['ColumnAmountO3'].isna().mean()

    # k 분포 확인 (Source_Latitude 중복 = 역방향 duplicate 지표)
    df_valid = df[df['Source_Latitude'].notna()]
    dup_count = df_valid['Source_Latitude'].duplicated().sum()

    print(f"[{year}-{month:02d}] shape={df.shape}, NaN={nan_rate:.4f}, "
          f"n_valid={df_valid.shape[0]}, Source_Lat 중복={dup_count}")